In [ ]:
pip install scgraph>=3.3.0

In this notebook we will be modifying the marnet geograph to clean up the ocean route in the Tsugaru Straight. 

The Tsugaru Straight is a narrow straight between the islands of Honshu and Hokkaido in Japan. The current route goes through the straight but crosses over some land. We will be modifying the route to go around the southern tip of Hokkaido.

First we will import the current marnet geograph and plot it. Then we will modify the route and plot it again.

In [ ]:
from scgraph import GeoGraph
marnet_geograph = GeoGraph.load_geograph("marnet")
# Save the geograph as a GeoJSON file in your current working directory
marnet_geograph.save_as_geojson("marnet_geograph.geojson")

Navigate to `geojson.io` and add the `marnet_geograph.geojson` file you just created to the map. 

Zoom in on the Tsugaru Straight and find the edge that crosses over the land. This is the edge we will be modifying. Click on the edge to view its properties. Take note of the `origin_idx` and `destination_idx`. These are the indices of the origin and destination nodes of the edge in the nodes array.

In the top right hand of the map, you can click on the little map pin icon to drop a new point on the map. Drop it off the southern tip of Hokkaido. This will be the new point that the edge will pass through. Then drop another point part way through the straight to avoid going over the eastern peninsula of Hokkaido.

Click on those points, navigate to the `info` tab in the popup, and take note of the `Latitute` and `Longitude` of the points.

Now we will modify the geograph.

In [ ]:
# The two nodes of the edge to replace. With scgraph 3.6, geojson.io shows them as 57 and 8697
# (`origin_idx` and `destination_idx`). scgraph 4.0 numbers the nodes in another order, and
# `node_ids` gives the number that each node had in 3.6.
node_ids = getattr(marnet_geograph, "node_ids", None)
if node_ids is None:
    origin_idx, destination_idx = 57, 8697
else:
    node_ids = list(node_ids)
    origin_idx, destination_idx = node_ids.index(57), node_ids.index(8697)

# Remove the existing edge between the original nodes, in both directions:
marnet_geograph.remove_edge(origin_id=origin_idx, destination_id=destination_idx, symmetric=True)

# Add the two new nodes:
# Note: The latitude and longitude values are the ones I got when dropping my pins in geojson.io
# Note: add_coord_node() returns the index of the newly added node. We will use this index to add the edges.
# Note: auto_edge=False adds the node alone. We add its edges ourselves below.
n1 = marnet_geograph.add_coord_node(
    coord_dict={"latitude": 41.6757, "longitude": 141.2373},
    auto_edge=False,
)
n2 = marnet_geograph.add_coord_node(
    coord_dict={"latitude": 41.3559, "longitude": 140.2468},
    auto_edge=False,
)

# Add the three new edges, in both directions. Each edge gets its great-circle length:
marnet_geograph.add_haversine_edge(origin_idx=origin_idx, destination_idx=n1)
marnet_geograph.add_haversine_edge(origin_idx=n1, destination_idx=n2)
marnet_geograph.add_haversine_edge(origin_idx=n2, destination_idx=destination_idx)

# Save the modified geograph as a GeoJSON file in your current working directory
marnet_geograph.save_as_geojson("marnet_geograph_modified.geojson")

Go back to `geojson.io` and click on the `New` button in the top left. Then drag and drop your `marnet_geograph_modified.geojson` file into the map. Zoom in on the Tsugaru Straight to see the new route.

To save this as a custom geograph and avoid this overhead each time, you should save the modified geograph as a new file.

In [ ]:
marnet_geograph.save_as_graphjson("custom_marnet.graphjson")
# Now you can load this modified geograph in the future using:
# modified_marnet_geograph = GeoGraph.load_from_graphjson("custom_marnet.graphjson")